In [4]:
import six
import h5py
import numpy as np
from pathlib import Path
from __future__ import print_function
from os.path import isfile,expanduser,join

In [ ]:
# Adopted from illustris_python.groupcat 
# https://github.com/illustristng/illustris_python/blob/master/illustris_python/groupcat.py
# and modified due to some missing fields in the data
def gcPath(basePath, chunkNum=0):
    """Return absolute path to a group catalog HDF5 file.
    Args:
        basePath: Base directory path (e.g., "/kaggle/input/tng300-z99")
        chunkNum: File chunk number (0 or 1 in your case)
    """
    # Your files follow the pattern: groups_099.{chunkNum}.hdf5
    filePath = f"{basePath}/groups_099.{chunkNum}.hdf5"
    
    if isfile(expanduser(filePath)):
        return filePath
    else:
        raise FileNotFoundError(f"Group catalog file not found: {filePath}")
        
def loadObjects(basePath, gName, nName, fields):
    """ Load either halo or subhalo information from the group catalog. """
    result = {}

    # make sure fields is not a single element
    if isinstance(fields, six.string_types):
        fields = [fields]

    # load header from first chunk
    with h5py.File(gcPath(basePath), 'r') as f:

        header = dict(f['Header'].attrs.items())

        if 'N'+nName+'_Total' not in header and nName == 'subgroups':
            nName = 'subhalos' # alternate convention

        result['count'] = np.int64(f['Header'].attrs['N' + nName + '_Total'])

        if not result['count']:
            print('warning: zero groups, empty return (snap=' + str(snapNum) + ').')
            return result

        # if fields not specified, load everything
        if not fields:
            fields = list(f[gName].keys())

        for field in fields:
            # verify existence
            if field not in f[gName].keys():
                raise Exception("Group catalog does not have requested field [" + field + "]!")

            # replace local length with global
            shape = list(f[gName][field].shape)
            shape[0] = result['count']

            # allocate within return dict
            result[field] = np.zeros(shape, dtype=f[gName][field].dtype)

    # loop over chunks
    wOffset = 0

    for i in range(header['NumFiles']):
        f = h5py.File(gcPath(basePath, i), 'r')

        if not f['Header'].attrs['N'+nName+'_ThisFile']:
            continue  # empty file chunk

        # loop over each requested field
        for field in fields:
            if field not in f[gName].keys():
                raise Exception("Group catalog does not have requested field [" + field + "]!")

            # shape and type
            shape = f[gName][field].shape

            # read data local to the current file
            if len(shape) == 1:
                result[field][wOffset:wOffset+shape[0]] = f[gName][field][0:shape[0]]
            else:
                result[field][wOffset:wOffset+shape[0], :] = f[gName][field][0:shape[0], :]

        wOffset += shape[0]
        f.close()

    # only a single field? then return the array instead of a single item dict
    if len(fields) == 1:
        return result[fields[0]]

    return result


def loadSubhalos(basePath, fields=None):
    """ Load all subhalo information from the entire group catalog for one snapshot
       (optionally restrict to a subset given by fields). """

    return loadObjects(basePath, "Subhalo", "subgroups", fields)


def loadHalos(basePath, fields=None):
    """ Load all halo information from the entire group catalog for one snapshot
       (optionally restrict to a subset given by fields). """

    return loadObjects(basePath, "Group", "groups", fields)


def loadHeader(basePath):
    """ Load the group catalog header. """
    with h5py.File(gcPath(basePath), 'r') as f:
        header = dict(f['Header'].attrs.items())

    return header

In [ ]:
import numpy as np
import pandas as pd

h = 0.6774
snapshot = 99

ROOT = '..'
base_path = f"/kaggle/input/tng300-z99"

cuts = {
    "minimum_log_stellar_mass": 9,
    "minimum_log_halo_mass": 11,
    "minimum_n_star_particles": 50
}

subhalo_fields = ["SubhaloPos", "SubhaloMassType", "SubhaloLenType", "SubhaloHalfmassRadType", 
                  "SubhaloVel", "SubhaloVmax", "SubhaloGrNr", "SubhaloStellarPhotometrics"]
halo_fields = ["Group_M_Crit200", "GroupFirstSub", "GroupPos", "GroupVel"]

subhalos = loadSubhalos(base_path, fields=subhalo_fields)
halos = loadHalos(base_path, fields=halo_fields)

subhalo_pos = subhalos["SubhaloPos"][:] / (h*1e3)
subhalo_stellarmass = subhalos["SubhaloMassType"][:,4]
subhalo_halomass = subhalos["SubhaloMassType"][:,1]
subhalo_n_stellar_particles = subhalos["SubhaloLenType"][:,4]
subhalo_stellarhalfmassradius = subhalos["SubhaloHalfmassRadType"][:,4] 
subhalo_vel = subhalos["SubhaloVel"][:] 
subhalo_vmax = subhalos["SubhaloVmax"][:]
#subhalo_flag = subhalos["SubhaloFlag"][:]
subhalo_photometry = subhalos["SubhaloStellarPhotometrics"][:]
halo_id = subhalos["SubhaloGrNr"][:].astype(int)

halo_mass = halos["Group_M_Crit200"][:]
halo_primarysubhalo = halos["GroupFirstSub"][:].astype(int)
group_pos = halos["GroupPos"][:] / (h*1e3)
group_vel = halos["GroupVel"][:] 

halos = pd.DataFrame(
    np.column_stack(
        (np.arange(len(halo_mass)), group_pos, group_vel, 
         halo_mass, halo_primarysubhalo)
            ),
    columns=['halo_id', 'halo_x', 'halo_y', 'halo_z', 'halo_vx', 'halo_vy', 
             'halo_vz', 'halo_mass', 'halo_primarysubhalo']
)
halos['halo_id'] = halos['halo_id'].astype(int)
halos.set_index("halo_id", inplace=True)

subhalos = pd.DataFrame(
    np.column_stack(
        [halo_id, np.arange(len(subhalo_stellarmass)), subhalo_pos, 
         subhalo_vel, subhalo_n_stellar_particles, subhalo_stellarmass, subhalo_halomass, 
         subhalo_stellarhalfmassradius, subhalo_vmax, subhalo_photometry]
        ),
        columns=['halo_id', 'subhalo_id', 'subhalo_x', 
                 'subhalo_y', 'subhalo_z', 'subhalo_vx', 'subhalo_vy', 
                 'subhalo_vz', 'subhalo_n_stellar_particles', 'subhalo_stellarmass', 
                 'subhalo_halomass', 'subhalo_stellarhalfmassradius', 'subhalo_vmax', 
                 'subhalo_photo_U', "subhalo_photo_B", "subhalo_photo_V", 
                 "subhalo_photo_K", "subhalo_photo_g", "subhalo_photo_r", 
                 "subhalo_photo_i", "subhalo_photo_z"],
    )

subhalos["is_central"] = (halos.loc[subhalos.halo_id]["halo_primarysubhalo"].values == subhalos["subhalo_id"].values)


#subhalos = subhalos[subhalos["subhalo_flag"] != 0].copy()
subhalos['halo_id'] = subhalos['halo_id'].astype(int)
subhalos['subhalo_id'] = subhalos['subhalo_id'].astype(int)

subhalos["subhalo_logstellarmass"] = np.log10(subhalos["subhalo_stellarmass"] / h)+10
subhalos["subhalo_loghalomass"] = np.log10(subhalos["subhalo_halomass"] / h)+10
subhalos["subhalo_logvmax"] = np.log10(subhalos["subhalo_vmax"])
subhalos["subhalo_logstellarhalfmassradius"] = np.log10(subhalos["subhalo_stellarhalfmassradius"])

#subhalos.drop("subhalo_flag", axis=1, inplace=True)
subhalos = subhalos[subhalos["subhalo_loghalomass"] > cuts["minimum_log_halo_mass"]].copy()
# stellar mass and particle cuts
subhalos = subhalos[subhalos["subhalo_n_stellar_particles"] > cuts["minimum_n_star_particles"]].copy()
subhalos = subhalos[subhalos["subhalo_logstellarmass"] > cuts["minimum_log_stellar_mass"]].copy()

subhalos.to_parquet(f'/kaggle/working/TNG300-1-subhalos_{snapshot}.parquet')
halos.to_parquet(f'/kaggle/working/TNG300-1-halos_{snapshot}.parquet')